# 05 – Amino-acid enrichment against a background population

**Question:** which amino acids are over- or under-represented in a group of peptides (Natural, Synthetic, Gram+, Gram-, Fungus, Cancer, Virus, Mammalian-active) compared with the background?

**Input:** the **non-redundant** peptides from notebook 02 (CD-HIT representatives, 90% identity). Redundancy removal matters most here: a large family of near-identical peptides would otherwise dominate the residue counts of a group.

## Background population
An enrichment is only meaningful relative to a reference. Our **background population** is *all other non-redundant DBAASP peptides*, i.e. every peptide **outside** the group being tested. (Using the whole population *including* the group would hide differences for big groups: Synthetic peptides are 76% of all peptides, so they could never look very different from "everything". The version against the whole population is kept in the results table as `log2_vs_all`.)

For each group and each of the 20 amino acids:

* `pct_group` = share of the group's residues that are this amino acid
* `pct_background` = the same share among the residues of all peptides outside the group
* **log2 fold enrichment** = log2(`pct_group` / `pct_background`). 0 means "as common as in the background", +1 means twice as common, −1 means half as common.

## Statistics
* **Test:** for each amino acid, Fisher's exact test on residue counts (group vs background), corrected across the 20 amino acids with Benjamini-Hochberg (q-value).
* **Uncertainty:** residues in the same peptide are not independent, so the plain Fisher p-values are too optimistic. We therefore also **bootstrap over peptides** (resample all peptides with replacement 1,000 times) to get a 95% confidence interval for the log2 enrichment.
* **Effect size:** with thousands of residues even tiny differences are statistically significant, so we also require a change of at least **1.5-fold** (|log2 enrichment| ≥ 0.58), a common convention.
* An amino acid is called **enriched / depleted** only if q < 0.05, the bootstrap interval excludes 0 **and** the change is at least 1.5-fold.
* **Negative control:** we run the same procedure on randomly chosen groups. If the method is sound, almost no amino acid should be called.

Groups overlap (a peptide can be Gram+ and Gram- and Cancer). That is fine here because every group is compared separately with its own background.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

df = pd.read_csv("../results/peptide_features.csv")
nr = df[df["is_representative"]].reset_index(drop=True)
print(f"{len(df)} cleaned peptides -> {len(nr)} non-redundant peptides used here")

AA = list("ACDEFGHIKLMNPQRSTVWY")
# amino acids ordered by chemical class for plotting: basic | acidic | polar | special | hydrophobic / aromatic
AA_PLOT = list("KRH") + list("DE") + list("STNQ") + list("CGP") + list("AVLIM") + list("FWY")
assert sorted(AA_PLOT) == sorted(AA)
MIN_LOG2 = np.log2(1.5)      # minimum effect size: 1.5-fold

# residue counts per peptide: N peptides x 20 amino acids
counts = np.array([[s.count(a) for a in AA] for s in nr["sequence"]])
assert (counts.sum(axis=1) == nr["length"].to_numpy()).all()
counts.shape

18196 cleaned peptides -> 9622 non-redundant peptides used here


(9622, 20)

## Groups

In [2]:
groups = {
    "Natural": nr["origin"] == "Natural",
    "Synthetic": nr["origin"] == "Synthetic",
    "Gram+": nr["is_gram_pos"] == 1,
    "Gram-": nr["is_gram_neg"] == 1,
    "Fungus": nr["is_fungus"] == 1,
    "Cancer": nr["is_cancer"] == 1,
    "Virus": nr["is_virus"] == 1,
    "Mammalian-active": nr["is_mammalian_cell"] == 1,
}
groups = {k: v.to_numpy() for k, v in groups.items()}
pd.DataFrame({"peptides": {k: int(v.sum()) for k, v in groups.items()},
              "% of all peptides": {k: round(100 * v.mean(), 1) for k, v in groups.items()},
              "residues": {k: int(counts[v].sum()) for k, v in groups.items()}})

,peptides,% of all peptides,residues
Natural,2308,24.0,64988
Synthetic,7314,76.0,121271
Gram+,7603,79.0,149761
Gram-,7779,80.8,153201
Fungus,2859,29.7,59693
Cancer,1813,18.8,31985
Virus,654,6.8,13752
Mammalian-active,5327,55.4,98234


## Composition of the whole population
For reference: amino-acid composition of all non-redundant peptides (pooled residues).

In [3]:
all_counts = counts.sum(axis=0)
population = pd.Series(100 * all_counts / all_counts.sum(), index=AA).loc[AA_PLOT]
population.round(2).rename("all peptides, %").to_frame().T

,K,R,H,D,E,S,T,N,Q,C,G,P,A,V,L,I,M,F,W,Y
"all peptides, %",14.28,9.73,2.06,1.83,2.07,4.25,2.91,2.69,2.3,3.65,7.73,3.84,7.05,5.54,11.07,6.41,1.08,5.08,4.3,2.12


## Enrichment function

In [4]:
def bh(p):
    """Benjamini-Hochberg adjusted p-values (q-values)."""
    p = np.asarray(p); n = len(p)
    order = np.argsort(p)
    q = np.empty(n)
    q[order] = np.minimum.accumulate((p[order] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1)

def log2_ratio(a, b):
    """log2 of (share of a) / (share of b) per amino acid; a 0.5 pseudo-count avoids log(0)."""
    return np.log2(((a + 0.5) / (a.sum() + 10)) / ((b + 0.5) / (b.sum() + 10)))

def enrichment(mask, n_boot=1000, seed=0):
    fg, bgc, everything = counts[mask].sum(0), counts[~mask].sum(0), counts.sum(0)
    lfe = log2_ratio(fg, bgc)                      # group vs background (peptides outside the group)

    # Fisher's exact test per amino acid: group residues vs background residues
    p = np.array([fisher_exact([[fg[i], fg.sum() - fg[i]], [bgc[i], bgc.sum() - bgc[i]]])[1] for i in range(20)])
    q = bh(p)

    # bootstrap over peptides -> confidence interval for the log2 enrichment
    rng = np.random.default_rng(seed)
    N = len(mask)
    boot = np.empty((n_boot, 20))
    for b in range(n_boot):
        w = np.bincount(rng.integers(0, N, N), minlength=N)     # how often each peptide is drawn
        boot[b] = log2_ratio((w * mask) @ counts, (w * ~mask) @ counts)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

    return pd.DataFrame({
        "aa": AA, "n_peptides": int(mask.sum()), "n_residues": int(fg.sum()),
        "pct_group": 100 * fg / fg.sum(), "pct_background": 100 * bgc / bgc.sum(),
        "log2_enrichment": lfe, "ci_low": lo, "ci_high": hi,
        "log2_vs_all": log2_ratio(fg, everything),   # for reference: against the whole population incl. the group
        "p_fisher": p, "q_bh": q,
        "significant": (q < 0.05) & ((lo > 0) | (hi < 0)) & (np.abs(lfe) >= MIN_LOG2),
    })

# tiny checks: identical composition -> 0, and a group twice as rich in K -> +1
assert np.allclose(log2_ratio(all_counts, all_counts), 0)
a = np.array([200] + [100] * 19); b = np.array([100] + [100] * 19)
assert abs(log2_ratio(a, b)[0] - np.log2((200 / 2100) / (100 / 2000))) < 0.01
print("ok")

ok


## Run for all groups

In [5]:
res = []
for name, mask in groups.items():
    r = enrichment(mask)
    r.insert(0, "group", name)
    res.append(r)
res = pd.concat(res, ignore_index=True)
res["direction"] = np.where(~res["significant"], "", np.where(res["log2_enrichment"] > 0, "enriched", "depleted"))
res.to_csv("../results/enrichment_results.csv", index=False)
population.rename("pct_all_peptides").to_frame().to_csv("../results/enrichment_population.csv")

print("amino acids called enriched / depleted per group (out of 20):")
tab = res.groupby("group")["direction"].value_counts().unstack(fill_value=0)
tab.reindex(list(groups)).reindex(columns=["enriched", "depleted"], fill_value=0)

amino acids called enriched / depleted per group (out of 20):


direction,enriched,depleted
group,,
Natural,7,3
Synthetic,3,7
Gram+,0,2
Gram-,0,2
Fungus,1,1
Cancer,0,0
Virus,5,3
Mammalian-active,0,4


## Negative control: random groups
100 random groups of the same size as Natural and as Gram+, 200 bootstraps each. With no real difference, (almost) no amino acid should be called enriched or depleted.

In [6]:
rng = np.random.default_rng(42)
control = []
for label in ["Natural", "Gram+"]:
    size = int(groups[label].sum())
    for i in range(100):
        m = np.zeros(len(nr), bool); m[rng.choice(len(nr), size, replace=False)] = True
        control.append({"same_size_as": label, "n_called": int(enrichment(m, n_boot=200, seed=i)["significant"].sum())})
control = pd.DataFrame(control)
ctrl = control.groupby("same_size_as")["n_called"].agg(["mean", "max"])
ctrl.to_csv("../results/enrichment_negative_control.csv")
print("amino acids called (out of 20) per random group:")
ctrl

amino acids called (out of 20) per random group:


,mean,max
same_size_as,,
Gram+,0.0,0
Natural,0.0,0


## Heatmap: enrichment of every amino acid in every group
Colour = log2 enrichment relative to the background (red = more common in the group, blue = less common). A star marks amino acids that meet the full rule (q < 0.05, bootstrap interval excludes 0, at least 1.5-fold).

In [7]:
mat = res.pivot(index="group", columns="aa", values="log2_enrichment").loc[list(groups), AA_PLOT]
sig = res.pivot(index="group", columns="aa", values="significant").loc[list(groups), AA_PLOT]
npep = res.groupby("group")["n_peptides"].first().loc[list(groups)]

fig, ax = plt.subplots(figsize=(10, 4.2))
im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(20)); ax.set_xticklabels(AA_PLOT)
ax.set_yticks(range(len(groups))); ax.set_yticklabels([f"{g} (n={npep[g]})" for g in groups])
for i in range(mat.shape[0]):
    for j in range(20):
        if sig.values[i, j]:
            ax.text(j, i, "*", ha="center", va="center", fontsize=13)
for x in [2.5, 4.5, 8.5, 11.5, 16.5]:
    ax.axvline(x, color="white", lw=2)
ax.set_title("Amino-acid enrichment vs background (log2 fold; * = significant and >= 1.5-fold)", fontsize=10)
fig.colorbar(im, ax=ax, label="log2 enrichment", shrink=0.85)
plt.tight_layout()
plt.savefig("../figures/enrichment_heatmap.png", dpi=150)
plt.show()

## Natural vs Synthetic, with confidence intervals
Enrichment in **Natural** peptides, with the background being all Synthetic peptides (the only peptides outside the Natural group). Positive = more common in Natural, negative = more common in Synthetic. Bars are the bootstrap 95% intervals; filled dots meet the full rule.

In [8]:
r = res[res.group == "Natural"].set_index("aa").loc[AA_PLOT]
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(20)
ax.errorbar(x, r["log2_enrichment"], yerr=[r["log2_enrichment"] - r["ci_low"], r["ci_high"] - r["log2_enrichment"]],
            fmt="none", ecolor="grey", capsize=2, zorder=1)
ax.scatter(x, r["log2_enrichment"], s=28, zorder=2,
           c=np.where(r["log2_enrichment"] > 0, "#2a9d8f", "#e76f51"),
           edgecolors=np.where(r["significant"], "black", "none"), linewidths=1)
for v in [MIN_LOG2, -MIN_LOG2]:
    ax.axhline(v, color="black", lw=0.6, ls=":")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(AA_PLOT)
ax.set_ylabel("log2( Natural / Synthetic )")
ax.set_title("Natural vs Synthetic amino-acid enrichment (dotted lines = 1.5-fold)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../figures/enrichment_natural_vs_synthetic.png", dpi=150)
plt.show()

## Strongest enrichments per group
The three most enriched and three most depleted amino acids per group (only those meeting the full rule).

In [9]:
def fmt(r):
    return ", ".join(f"{a} ({v:+.2f})" for a, v in zip(r.aa, r.log2_enrichment))
rows = []
for g in groups:
    s = res[(res.group == g) & res.significant]
    rows.append({"group": g,
                 "most enriched": fmt(s[s.log2_enrichment > 0].nlargest(3, "log2_enrichment")) or "-",
                 "most depleted": fmt(s[s.log2_enrichment < 0].nsmallest(3, "log2_enrichment")) or "-"})
top = pd.DataFrame(rows).set_index("group")
top.to_csv("../results/enrichment_top.csv")
top

,most enriched,most depleted
group,,
Natural,"C (+1.66), D (+0.94), G (+0.92)","W (-1.82), R (-1.02), K (-0.74)"
Synthetic,"W (+1.82), R (+1.02), K (+0.74)","C (-1.66), D (-0.94), G (-0.92)"
Gram+,-,"E (-0.96), Q (-0.63)"
Gram-,-,"E (-1.07), D (-0.62)"
Fungus,C (+0.70),W (-0.75)
Cancer,-,-
Virus,"E (+1.77), Q (+1.18), D (+1.12)","P (-0.77), K (-0.68), R (-0.65)"
Mammalian-active,-,"E (-0.86), Y (-0.66), C (-0.65)"


## Natural vs Synthetic: residue table

In [10]:
n = res[res.group == "Natural"].set_index("aa").loc[AA_PLOT]
tab = pd.DataFrame({"pct_Natural": n["pct_group"], "pct_Synthetic": n["pct_background"],
                    "log2(Natural/Synthetic)": n["log2_enrichment"], "ci_low": n["ci_low"], "ci_high": n["ci_high"],
                    "q_bh": n["q_bh"], "call": n["direction"].replace({"enriched": "more in Natural", "depleted": "more in Synthetic"})})
tab.round(3)

,pct_Natural,pct_Synthetic,log2(Natural/Synthetic),ci_low,ci_high,q_bh,call
aa,,,,,,,
K,9.962,16.597,-0.736,-0.792,-0.681,0.000,more in Synthetic
R,5.836,11.819,-1.018,-1.100,-0.937,0.000,more in Synthetic
H,2.091,2.045,0.032,-0.103,0.160,0.505,
D,2.657,1.382,0.943,0.815,1.061,0.000,more in Natural
E,2.560,1.803,0.506,0.380,0.635,0.000,
S,5.638,3.509,0.684,0.608,0.763,0.000,more in Natural
T,3.899,2.386,0.709,0.620,0.810,0.000,more in Natural
N,3.681,2.164,0.766,0.674,0.858,0.000,more in Natural
Q,2.519,2.188,0.203,0.084,0.328,0.000,


# Findings and limitations

Background = all other non-redundant peptides (outside the group). "Enriched / depleted" means q < 0.05, bootstrap 95% interval excluding 0 and at least 1.5-fold. The negative control (200 random groups) called **no** amino acid in any group, so the rule does not produce false positives on random data.

## Natural vs Synthetic (2,308 vs 7,314 peptides)
Because the background of Natural is Synthetic (and the other way round), the two results are mirror images. Read as "Natural relative to Synthetic":
* **More common in natural peptides:** cysteine (6.6% vs 2.1% of residues, +1.66 log2, about 3.2-fold), aspartate (+0.94), glycine (11.1% vs 5.9%, +0.92), asparagine (+0.77), threonine (+0.71), serine (+0.68) and proline (+0.62). Cysteine enrichment fits natural peptides that are stabilised by disulfide bonds.
* **More common in synthetic peptides:** tryptophan (5.7% vs 1.6% of residues, 3.5-fold), arginine (11.8% vs 5.8%, 2.0-fold) and lysine (16.6% vs 10.0%, 1.7-fold). This is the "cationic and tryptophan-rich" pattern typical of designed peptides, and it agrees with the higher charge density found in notebook 04.
* Glutamate (+0.51), leucine (−0.45), methionine (+0.36) and tyrosine (+0.30) differ in the same direction but by less than 1.5-fold, and histidine, isoleucine and valine barely differ.

## Target groups (each against the peptides outside the group)
* **Virus (n = 654):** enriched in glutamate (+1.77), glutamine (+1.18), aspartate (+1.12), threonine (+0.71) and asparagine (+0.67); depleted in proline (−0.77), lysine (−0.68) and arginine (−0.65). Virus-active peptides in DBAASP are therefore acidic and polar rather than cationic, which matches their median net charge of 0 in notebook 04.
* **Fungus (n = 2,859):** enriched in cysteine (+0.70), depleted in tryptophan (−0.75).
* **Gram+ (n = 7,603) and Gram- (n = 7,779):** no amino acid is enriched by 1.5-fold. Lysine and arginine are only modestly higher (Gram-: K +0.42, R +0.28; Gram+: K +0.11, R +0.15), while acidic residues are lower (Gram+: E −0.96, Q −0.63; Gram-: E −1.07, D −0.62). Because these two groups make up 79–81% of all peptides, their background is small and mostly virus-, cancer- or otherwise non-antibacterial peptides, so this partly mirrors the virus result. Against the whole population (column `log2_vs_all`) the differences are much smaller.
* **Cancer (n = 1,813):** no amino acid meets the rule; the largest changes are glutamate (−0.42) and histidine (−0.40).
* **Mammalian-active (n = 5,327):** depleted in glutamate (−0.86), tyrosine (−0.66), cysteine (−0.65) and aspartate (−0.64), i.e. lower in acidic residues; lysine (+0.44), tryptophan (+0.40) and leucine (+0.39) are higher but below 1.5-fold. This matches the small positive charge effect in notebook 04.

## Limitations
* Enrichment is computed on pooled residues, so long peptides contribute more residues than short ones. The bootstrap over peptides accounts for this in the confidence intervals, but not in the fold changes themselves.
* Composition is not corrected for length or for the position of a residue in the peptide.
* The background depends on the comparison: for large or heterogeneous groups (Gram+, Gram-) the "outside the group" peptides are a small, specific subset, so results should be read as "group vs the rest of DBAASP", not as absolute composition biases. Groups also overlap, so the target results are not independent of each other.
* The 1.5-fold cut-off is a convention; differences below it are reported in the notes but not called.
* A proteome-wide reference (for example Swiss-Prot amino-acid frequencies) was not used as an additional background.
* Redundancy removal with CD-HIT at 90% identity (notebook 02) is essential for this analysis, but it keeps the longest sequence of a cluster and merges fragments into longer peptides.